# Ataxx Zero - Kaggle v10

**Antes de Run All:**
1. Settings -> Accelerator -> **GPU T4 x2**.
2. Add-ons -> Secrets -> agregar `HF_TOKEN`.
3. Revisar solo la celda **Run config**.

El notebook se mantiene chico: clona `main`, arma `v10_pretrain.npz` desde replays versionadas, carga `kaggle/v10_config.json` y ejecuta `train.py` como subprocess.


In [ ]:
# === Run config (lo unico que deberia cambiar entre runs) ===

RUN_NAME = "policy_spatial_v10"
BASE_CONFIG_PATH = "kaggle/v10_config.json"

# Data curada versionada en el repo. Los buffers HF se pueden bajar antes
# con scripts/fetch_hf_buffers.py si queremos ampliar esta fuente.
CURATED_SOURCE = "tournament_replays"
CURATED_OUTPUT = "/kaggle/working/data/curated/v10_pretrain.npz"
HUMAN_OVERSAMPLE = 2.0

# Hardware Kaggle
TRAINER_DEVICES = 2
TRAINER_STRATEGY = "ddp_spawn"
TRAINER_PRECISION = "16-mixed"
SELFPLAY_WORKERS = 2

# Persistencia
HF_REPO_ID = "dieg0code/ataxx-zero"
KEEP_LAST_N_LOCAL = 3


In [ ]:
# === Clonar / actualizar el repo ===

REPO_URL = "https://github.com/Dieg0Code/ataxx-zero-ai.git"
BRANCH = "main"
WORKDIR = "/kaggle/working/ataxx-zero-ai"

import os, subprocess
from pathlib import Path

if not Path(WORKDIR).exists():
    subprocess.check_call(["git", "clone", REPO_URL, WORKDIR])
else:
    subprocess.check_call(["git", "-C", WORKDIR, "fetch", "--prune"])
subprocess.check_call(["git", "-C", WORKDIR, "checkout", BRANCH])
subprocess.check_call(["git", "-C", WORKDIR, "pull", "--ff-only"])

os.chdir(WORKDIR)
subprocess.check_call(["git", "-C", WORKDIR, "log", "-1", "--oneline"])

In [ ]:
# === Instalar dependencias (uv + grupo train) ===

!python -m pip install -q uv
!uv sync --frozen --group train
!uv run python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'devices', torch.cuda.device_count())"

In [ ]:
# === HF Token desde Kaggle Secrets ===

HF_ENABLED = False
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN").strip()
    if not token:
        raise RuntimeError("HF_TOKEN secret está vacío.")
    os.environ["HF_TOKEN"] = token
    HF_ENABLED = True
    print(f"HF_TOKEN cargado ({len(token)} chars). HF persistence ENABLED.")
except Exception as exc:
    print(f"Sin HF_TOKEN en Kaggle Secrets: {exc}")
    print("HF persistence DISABLED — los checkpoints solo van a quedar locales y se pierden al cerrar la sesión.")

In [ ]:
# === Dataset curado para pretraining ===

from pathlib import Path
import subprocess

Path(CURATED_OUTPUT).parent.mkdir(parents=True, exist_ok=True)
subprocess.check_call([
    "uv", "run", "python", "scripts/curate_training_data.py",
    "--source", CURATED_SOURCE,
    "--output", CURATED_OUTPUT,
    "--human-oversample", str(HUMAN_OVERSAMPLE),
])


In [ ]:
# === Detectar GPUs y escribir run_config.json ===

import json
import torch

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    names = [torch.cuda.get_device_name(i) for i in range(n_gpus)]
    print(f"GPUs detectadas: {n_gpus} ({', '.join(names)})")
    effective_devices = min(TRAINER_DEVICES, n_gpus)
    effective_strategy = TRAINER_STRATEGY if effective_devices > 1 else "auto"
else:
    print("Sin CUDA: revisa Accelerator en Settings antes de entrenar.")
    effective_devices = 1
    effective_strategy = "auto"

with open(BASE_CONFIG_PATH, "r", encoding="utf-8") as fh:
    run_config = json.load(fh)

run_config.update({
    "hf_enabled": HF_ENABLED,
    "hf_repo_id": HF_REPO_ID,
    "hf_run_id": RUN_NAME,
    "trainer_devices": effective_devices,
    "trainer_strategy": effective_strategy,
    "trainer_precision": TRAINER_PRECISION,
    "selfplay_workers": SELFPLAY_WORKERS,
    "pretrain_dataset_path": CURATED_OUTPUT,
    "hf_local_dir": "/kaggle/working/hf_checkpoints",
    "keep_last_n_local_checkpoints": KEEP_LAST_N_LOCAL,
    "keep_last_n_hf_checkpoints": KEEP_LAST_N_LOCAL,
})

config_path = "/kaggle/working/run_config.json"
with open(config_path, "w", encoding="utf-8") as fh:
    json.dump(run_config, fh, indent=2)

print(f"Run          : {RUN_NAME}")
print(f"Config       : {BASE_CONFIG_PATH} -> {config_path}")
print(f"Pretrain data: {CURATED_OUTPUT}")
print(f"Scope        : {run_config['iterations']} iters, {run_config['episodes_per_iter']} eps/iter, {run_config['mcts_sims']} sims")
print(f"Gate         : baseline={run_config['baseline_checkpoint']} composite={run_config['baseline_composite']} h2h>={run_config['baseline_h2h_min_score']}")
print(f"Hardware     : devices={effective_devices}, strategy={effective_strategy}, precision={TRAINER_PRECISION}")
print(f"HF           : {HF_REPO_ID if HF_ENABLED else 'disabled'}")


In [ ]:
# === Entrenar ===
# train.py corre como subprocess (no `import train`) para que ddp_spawn pueda
# re-ejecutar el script en cada worker. Llamar train.main() directo desde el
# notebook hace que los workers spawned pidan re-ejecutar el kernel launcher
# de Jupyter, que rechaza los args de Lightning con SystemExit: 2.

extra = ["--hf"] if HF_ENABLED else []
subprocess.check_call([
    "uv", "run", "python", "train.py",
    "--config-json", config_path,
    *extra,
])

In [ ]:
# === Post-run: historial a CSV ===

if HF_ENABLED:
    !uv run python scripts/fetch_run_history.py {RUN_NAME}
    print(f"CSV en runs_history/{RUN_NAME}/{RUN_NAME}_history.csv")
else:
    print("HF desactivado: no hay metadata remota para descargar.")
